In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


In [19]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas
# TARGETS = ["Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 6

In [20]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}"]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

# RNA

In [21]:
def PrepareData(VirtualDataset, OriginalDataset, target, i):
    X_orig = OriginalDataset[PREDICTORS].values
    Y_orig = OriginalDataset[target].values
    
    Xv = VirtualDataset[PREDICTORS].values
    Yv = VirtualDataset[target].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_orig, Y_orig, test_size=0.2, random_state=42
    )
    
    # CONCATENAÇÃO CORRETA
    X_train = np.concatenate((X_train, Xv), axis=0)
    Y_train = np.concatenate((Y_train, Yv), axis=0)

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)
    
    return x_train, x_test, Y_train, Y_test

    
def PrintDim(x, y):
    print(f"Dimensão da entrada: {np.shape(x)}")
    print(f"Dimensão da saida: {np.shape(y)}")

In [22]:
from sklearn.metrics import mean_squared_error, r2_score

def TrainANN(
    x_train, y_train,
    x_test, y_test,
    n_hidden=[],
    lr=1e-3,
    l2_reg=1e-4,
    epochs=500,
    batch_size=8,
    verbose=0
):
    n_inputs = x_train.shape[1]

    # ======================
    # Modelo
    # ======================
    model = Sequential()

    model.add(
        Dense(
            n_hidden[0],
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            input_shape=(n_inputs,)
        )
    )

    for units in n_hidden[1:]:
        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=l2(l2_reg)
            )
        )

    model.add(Dense(1, activation="linear"))

    w0 = model.get_weights()


    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )

    # ======================
    # Treinamento
    # ======================
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose
    )
    wf = model.get_weights()

    # ======================
    # Predições (NORMALIZADAS)
    # ======================
    y_train_pred_norm = model.predict(x_train, verbose=0)
    y_test_pred_norm  = model.predict(x_test,  verbose=0)

    # ======================
    # DESNORMALIZAÇÃO
    # ======================
    y_train_real = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_test_real  = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()

    y_train_pred = OUT_SCALER.inverse_transform(y_train_pred_norm).ravel()
    y_test_pred  = OUT_SCALER.inverse_transform(y_test_pred_norm).ravel()

    # ======================
    # MÉTRICAS NO ESPAÇO FÍSICO
    # ======================
    metrics = {
    "mse_train": round(mean_squared_error(y_train_real, y_train_pred), 4),
    "mse_test":  round(mean_squared_error(y_test_real,  y_test_pred), 4),
    "r2_train":  round(r2_score(y_train_real, y_train_pred), 4),
    "r2_test":   round(r2_score(y_test_real,  y_test_pred), 4)
}


    return model, history, metrics, w0, wf


In [23]:
Datasets[1:]

[      pH   Cond   Temp    OD    Tds   Resist  Salin    ORP     IP   Cor  \
 27  6.73  114.0  29.50  2.56   57.0   8770.0   0.05  111.2  15.60  8.40   
 28  7.46  132.0  30.31  1.24   66.0   7580.0   0.06  178.9   8.80  7.70   
 29  6.69  216.0  29.77  1.07  108.0   4630.0   0.10  192.5  72.00  7.80   
 30  6.83  191.0  29.80  0.61   95.0   5236.0   0.09  154.6  28.40  6.30   
 31  6.51  109.0  28.62  2.81   55.0   9170.0   0.05  133.7  85.60  7.90   
 32  6.21   80.0  27.11  1.66   40.0  12500.0   0.04  146.0  69.00  9.30   
 33  6.26   87.0  28.11  1.34   44.0  11490.0   0.04  136.9  18.60  8.30   
 34  6.22   90.0  28.35  1.18   45.0  11110.0   0.04  190.8   1.20  7.10   
 35  6.48  111.0  29.66  1.75   56.0   9010.0   0.05  118.1  11.60  6.90   
 36  6.58  155.0  28.84  2.17   77.0   6452.0   0.07  130.9  32.40  6.70   
 37  6.66  270.0  29.36  2.67  135.0   3704.0   0.13  132.8  23.40  7.20   
 38  6.49  193.0  27.48  1.02   97.0   5181.0   0.09   97.5  40.60  6.70   
 39  6.65  1

In [25]:
neurons = [[1], [2], [4], [6], [8], [10], [14], [16], [18], [20]]
all_metrics = []

i=1
for Dataset in Datasets[1:]:
    os.makedirs(f"./Dados/VirtualData/P{i+1}/FilterResults/", exist_ok=True)

    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")
        output_dir = f"./Dados/VirtualData/P{i+1}"
        vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
        VirtualDataset = pd.read_excel(vs_filename, sheet_name="orig-vs")
        x_train, x_test, y_train, y_test = PrepareData(VirtualDataset, Dataset, target, i)
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        PrintDim(x_train, y_train)
        PrintDim(x_test, y_test)
        
        
        for neuron in neurons:
            for k in range(10):
                model, history, metrics, w0, wf = TrainANN(
                    x_train, y_train,
                    x_test, y_test,
                    n_hidden=neuron
                )

                # adiciona metadados
                metrics.update({
                    "P": i + 1,
                    "runTime": k,
                    "target": target,
                    "neurons": neuron[0],  # salva como int
                    "W0": str([w.round(4).tolist() for w in w0]),
                    "Wf": str([w.round(4).tolist() for w in wf]),
                })

                # cria DataFrame da linha atual
                df_new = pd.DataFrame([metrics])

                excel_file = "Results-vs.xlsx"
                # salva incrementalmente no Excel
                if os.path.exists(excel_file):
                    df_old = pd.read_excel(excel_file)
                    df_final = pd.concat([df_old, df_new], ignore_index=True)
                else:
                    df_final = df_new

                df_final.to_excel(excel_file, index=False)

                print(f"Modelo P{i+1}_{target}_{neuron}_{k} treinado e salvo no Excel")
    i = i + 1

 → Fe
Dimensão da entrada: (230, 10)
Dimensão da saida: (230,)
Dimensão da entrada: (6, 10)
Dimensão da saida: (6,)
Modelo P2_Fe_[1]_0 treinado e salvo no Excel
Modelo P2_Fe_[1]_1 treinado e salvo no Excel
Modelo P2_Fe_[1]_2 treinado e salvo no Excel
Modelo P2_Fe_[1]_3 treinado e salvo no Excel
Modelo P2_Fe_[1]_4 treinado e salvo no Excel
Modelo P2_Fe_[1]_5 treinado e salvo no Excel
Modelo P2_Fe_[1]_6 treinado e salvo no Excel
Modelo P2_Fe_[1]_7 treinado e salvo no Excel
Modelo P2_Fe_[1]_8 treinado e salvo no Excel
Modelo P2_Fe_[1]_9 treinado e salvo no Excel
Modelo P2_Fe_[2]_0 treinado e salvo no Excel
Modelo P2_Fe_[2]_1 treinado e salvo no Excel
Modelo P2_Fe_[2]_2 treinado e salvo no Excel
Modelo P2_Fe_[2]_3 treinado e salvo no Excel
Modelo P2_Fe_[2]_4 treinado e salvo no Excel
Modelo P2_Fe_[2]_5 treinado e salvo no Excel
Modelo P2_Fe_[2]_6 treinado e salvo no Excel
Modelo P2_Fe_[2]_7 treinado e salvo no Excel
Modelo P2_Fe_[2]_8 treinado e salvo no Excel
Modelo P2_Fe_[2]_9 treinado e